# pdf2md no Colab (LightOnOCR-2 + llama.cpp)

Converte PDFs em Markdown com o `tools/pdf2md.py` do repositório, usando vários processos `llama-server` na GPU do Colab.

**Antes de rodar**
1. Ambiente de execução: GPU (testado numa A100; L4 e T4 também servem).
2. No Google Drive, crie `Colab Notebooks/PDF2MD/` com:
   - `pdf2md.py` (copie de `tools/pdf2md.py` do repositório, **sempre a versão mais recente**)
   - `pdfs/` com os PDFs a converter
3. Rode as células na ordem. Os resultados vão para `PDF2MD/out/`.

Na primeira vez a célula 3 compila o llama.cpp (~10 a 15 min) e guarda o binário no Drive. Nas próximas sessões ela só o restaura, desde que a GPU tenha a mesma arquitetura.

## 1. GPU e Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/PDF2MD'
!ls -la "{DRIVE_DIR}" "{DRIVE_DIR}/pdfs"

## 2. Bibliotecas Python do pdf2md

In [ ]:
!pip install -q pypdfium2 Pillow PyPDF2 rich python-dotenv huggingface_hub mistralai

## 3. llama-server (restaura do Drive ou compila)

O cache só vale para a mesma arquitetura de GPU (`compute_cap`); se mudar, o binário é recompilado.

In [ ]:
import os, subprocess, shutil

BIN_CACHE = f'{DRIVE_DIR}/llama-server-bin'      # cache no Drive
LLAMA_BIN_DIR = '/content/llama-bin'             # cópia local, usada na execução
LLAMA_SERVER = f'{LLAMA_BIN_DIR}/llama-server'
BUILD_BIN = '/content/llama.cpp/build/bin'

cap = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']
).decode().split()[0].replace('.', '')

def copy_bins(src, dst):
    os.makedirs(dst, exist_ok=True)
    for name in os.listdir(src):
        path = os.path.join(src, name)
        if os.path.isfile(path) and (name == 'llama-server' or '.so' in name):
            shutil.copy2(path, dst)

arch_file = f'{BIN_CACHE}/arch.txt'
cache_ok = (
    os.path.exists(f'{BIN_CACHE}/llama-server')
    and os.path.exists(arch_file)
    and open(arch_file).read().strip() == cap
)

if cache_ok:
    copy_bins(BIN_CACHE, LLAMA_BIN_DIR)
    print(f'binário restaurado do Drive (CUDA arch {cap})')
else:
    print(f'cache ausente ou de outra GPU; compilando para CUDA arch {cap}')
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON \
        -DCMAKE_CUDA_ARCHITECTURES={cap} -DCMAKE_BUILD_TYPE=Release \
        -DLLAMA_CURL=OFF -DBUILD_SHARED_LIBS=ON
    !cmake --build /content/llama.cpp/build -j 8 --target llama-server
    copy_bins(BUILD_BIN, LLAMA_BIN_DIR)
    shutil.rmtree(BIN_CACHE, ignore_errors=True)
    copy_bins(BUILD_BIN, BIN_CACHE)
    open(arch_file, 'w').write(cap)
    print('binário salvo no Drive')

os.environ['LD_LIBRARY_PATH'] = f'{LLAMA_BIN_DIR}:' + os.environ.get('LD_LIBRARY_PATH', '')
os.chmod(LLAMA_SERVER, 0o755)
!{LLAMA_SERVER} --version

## 4. Pesos do modelo

Use o projetor de visão **Q8_0**: o f16 gera lixo em CUDA.

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_DIR = '/content/models/lightonocr2'
for f in ['LightOnOCR-2-1B-f16.gguf', 'mmproj-LightOnOCR-2-1B-Q8_0.gguf']:
    hf_hub_download('ggml-org/LightOnOCR-2-1B-GGUF', f, local_dir=MODEL_DIR)
MODEL = f'{MODEL_DIR}/LightOnOCR-2-1B-f16.gguf'
MMPROJ = f'{MODEL_DIR}/mmproj-LightOnOCR-2-1B-Q8_0.gguf'
!ls -lh {MODEL_DIR}

## 5. Servidores

Um único `llama-server` fica limitado pela CPU; vários processos escalam melhor. Cada um usa ~9 GB de VRAM com 4 slots.

In [ ]:
import subprocess, time, urllib.request

NS = 4                      # número de processos llama-server
NP_EACH = 4                 # slots por processo
CTX_PER_SLOT = 16384        # imagem (~2400 tokens) + até 6144 de saída
PORTS = [8093 + i for i in range(NS)]

subprocess.run(['pkill', '-f', 'llama-server']); time.sleep(2)

servers = [subprocess.Popen(
    [LLAMA_SERVER, '-m', MODEL, '--mmproj', MMPROJ,
     '-c', str(NP_EACH * CTX_PER_SLOT), '-np', str(NP_EACH),
     '-ngl', '99', '--port', str(port)],
    stdout=open(f'/content/llama-server-{port}.log', 'w'), stderr=subprocess.STDOUT,
    start_new_session=True) for port in PORTS]

t0 = time.time()
for server, port in zip(servers, PORTS):
    while True:
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=2); break
        except Exception:
            if server.poll() is not None:
                print(open(f'/content/llama-server-{port}.log').read()[-3000:])
                raise RuntimeError(f'servidor da porta {port} encerrou')
            if time.time() - t0 > 180:
                raise RuntimeError(f'porta {port} não subiu em 180 s')
            time.sleep(2)

SERVER_URLS = ','.join(f'http://127.0.0.1:{p}/v1' for p in PORTS)
WORKERS = NS * NP_EACH
print(f'{NS} servidores no ar em {time.time() - t0:.0f}s, {WORKERS} slots no total')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader
print('URLs:', SERVER_URLS)

## 6. Converter os PDFs

O `.md` sai ao lado do PDF e é movido para `out/`. Em PDFs pequenos o tempo é dominado pela latência de cada onda de páginas; o ganho do paralelismo aparece nos PDFs grandes.

In [ ]:
import glob

OUT = f'{DRIVE_DIR}/out'
os.makedirs(OUT, exist_ok=True)

for pdf in sorted(glob.glob(f'{DRIVE_DIR}/pdfs/*.pdf')):
    print('\n===', os.path.basename(pdf))
    !python "{DRIVE_DIR}/pdf2md.py" "{pdf}" --model lighton --server-url "{SERVER_URLS}" --workers {WORKERS} -y
    md = os.path.splitext(pdf)[0] + '.md'
    if os.path.exists(md):
        os.replace(md, f'{OUT}/{os.path.basename(md)}')

## 7. Encerrar os servidores

Rode ao terminar para liberar a GPU. Interromper uma célula **não** derruba os servidores (eles rodam em sessão separada).

Evite `!pkill -f ...`: o shell do `!` casa com o próprio padrão e se mata.

In [ ]:
for s in servers:
    s.terminate()
subprocess.run(['pkill', '-f', 'llama-server'])